In [36]:
import yfinance
import numpy as np
import pandas as pd
from tqdm import tqdm

## Sample

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import json
import time
import os
from datetime import datetime

def ticker_to_json(ticker_symbol, outdir="./yf_dumps", sleep_sec=0.5):
    """
    Extracts all accessible data from a yfinance.Ticker object and dumps to JSON.
    Returns the combined dictionary (also written to disk).
    """
    os.makedirs(outdir, exist_ok=True)
    t = yf.Ticker(ticker_symbol)
    dump = {"ticker": ticker_symbol.upper(), "asof_utc": datetime.now().isoformat()}

    # Simple helper to convert DataFrames safely
    def safe_df(obj):
        if isinstance(obj, pd.DataFrame):
            obj.columns = [str(col) for col in obj.columns]
            return obj.reset_index().to_dict(orient="records")
        elif isinstance(obj, pd.Series):
            return obj.to_dict()
        elif obj is None:
            return None
        else:
            return str(obj)

    def convert_dict_keys_to_str(d):
        return {k.strftime("%Y-%m-%d"): v for k, v in d.items()}

    # --- Try each attribute defensively ---
    ticker_attr = ['actions', 'balance_sheet', 'capital_gains', 'cash_flow', 'earnings_dates', 'financials', 'history_metadata', 'income_stmt', 'info', 
    'news', 'mutualfund_holders', 'major_holders', 'isin', 'options', 'quarterly_balance_sheet', 'quarterly_cash_flow', 'quarterly_financials', 
    'quarterly_income_stmt', 'sec_filings', 'splits']

    #conversion_list = ['balance_sheet', 'cash_flow', 'financials', 'income_stmt', 'quarterly_balance_sheet', 'quarterly_cash_flow', 'quarterly_financials', 'quarterly_income_stmt']
    for a in ticker_attr:
        try:
            val = getattr(t, a)
            val = safe_df(val)
            if type(val) == dict:
                val = convert_dict_keys_to_str(val)
            dump[a] = val
        except Exception as e:
            dump[a] = f"Error: {type(e).__name__}: {e}"

    # --- Option chains (can be large, skip or limit) ---
    option_dates = []
    try:
        option_dates = t.options or []
    except Exception:
        option_dates = []

    chains = {}
    for d in option_dates[:3]:  # limit to first 3 expiries to avoid massive files
        try:
            oc = t.option_chain(d)
            chains[d] = {
                "calls": safe_df(oc.calls),
                "puts": safe_df(oc.puts)
            }
            time.sleep(0.2)
        except Exception as e:
            chains[d] = f"Error: {type(e).__name__}: {e}"
    dump["option_chains"] = chains

    # --- Recent price history (1mo daily, 5y weekly, 10y monthly) ---
    try:
        dump["history_5y_daily"] = safe_df(t.history(period="5y", interval="1d"))
        dump["history_5y_weekly"] = safe_df(t.history(period="5y", interval="1wk"))
        dump["history_10y_monthly"] = safe_df(t.history(period="10y", interval="1mo"))
    except Exception as e:
        dump["history_error"] = str(e)

    outpath = os.path.join(outdir, f"{ticker_symbol.upper()}_dump.json")
    with open(outpath, "w", encoding="utf-8") as f:
        json.dump(dump, f, indent=2, default=str)

    print(f"[{ticker_symbol}] written to {outpath}")
    time.sleep(sleep_sec)
    return dump

def json_loader(json_path):
    """
    Loads a JSON dump created by ticker_to_json.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

In [7]:
data_path = "./yf_dumps"

In [34]:
existing_json = [file for file in  os.listdir(data_path) if file.endswith(".json")]
existing_ticker = [file.replace("_dump.json", "") for file in existing_json]

# Load metadata and prepare ticker list
metadata = pd.read_csv('data/metadata.csv', encoding='cp1252')
ticker_list = metadata.Symbol.values.tolist()
print(len(ticker_list))
replace = False 

if not replace: 
    ticker_list = [t for t in ticker_list if t not in existing_ticker]

print(f"Total tickers to process: {len(ticker_list)}")
print(f'Total existing tickers: {len(existing_ticker)}')

503
Total tickers to process: 300
Total existing tickers: 203


In [ ]:
for ticker in tqdm(ticker_list):
    dump = ticker_to_json(ticker, outdir=data_path, sleep_sec=3)

  0%|          | 0/300 [00:00<?, ?it/s]

[FOX] written to ./yf_dumps\FOX_dump.json


  0%|          | 1/300 [00:06<32:20,  6.49s/it]

[BEN] written to ./yf_dumps\BEN_dump.json


  1%|          | 2/300 [00:14<37:23,  7.53s/it]

...

 30%|██▉       | 89/300 [11:50<27:26,  7.80s/it]

[LMT] written to ./yf_dumps\LMT_dump.json


 30%|███       | 90/300 [11:59<27:44,  7.93s/it]

[L] written to ./yf_dumps\L_dump.json


 30%|███       | 91/300 [12:06<27:13,  7.82s/it]

In [ ]:
data = {}
for file in existing_json:
    data[file.replace("_dump.json", "")] = json_loader(os.path.join(data_path, file))
    
price_history = list() 
for ticker in data.keys():
    df = pd.DataFrame(data[ticker]["history_5y_daily"])
    df["ticker"] = data[ticker]["ticker"]
    price_history.append(df)

price_history_df = pd.concat(price_history, ignore_index=True)
price_history_df

datetime.datetime(2025, 12, 3, 19, 44, 45, 227211)

In [ ]:
ticker_symbol = 'MMM'
outdir = './Test'
sleep_sec = 0.5

for k in dump.keys():
    print(k)
    outpath = os.path.join(outdir, f"{ticker_symbol.upper()}_dump_{k}.json")
    with open(outpath, "w", encoding="utf-8") as f:
        json.dump(dump[k], f, indent=2, default=str)

    print(f"[{ticker_symbol}] written to {outpath}")
    time.sleep(sleep_sec)

ticker
[MMM] written to ./yf_dumps\MMM_dump_ticker.json
asof_utc
[MMM] written to ./yf_dumps\MMM_dump_asof_utc.json
actions
[MMM] written to ./yf_dumps\MMM_dump_actions.json
balance_sheet
[MMM] written to ./yf_dumps\MMM_dump_balance_sheet.json
capital_gains
[MMM] written to ./yf_dumps\MMM_dump_capital_gains.json
cash_flow
[MMM] written to ./yf_dumps\MMM_dump_cash_flow.json
earnings_dates
[MMM] written to ./yf_dumps\MMM_dump_earnings_dates.json
financials
[MMM] written to ./yf_dumps\MMM_dump_financials.json
history_metadata
[MMM] written to ./yf_dumps\MMM_dump_history_metadata.json
income_stmt
[MMM] written to ./yf_dumps\MMM_dump_income_stmt.json
info
[MMM] written to ./yf_dumps\MMM_dump_info.json
news
[MMM] written to ./yf_dumps\MMM_dump_news.json
mutualfund_holders
[MMM] written to ./yf_dumps\MMM_dump_mutualfund_holders.json
major_holders
[MMM] written to ./yf_dumps\MMM_dump_major_holders.json
isin
[MMM] written to ./yf_dumps\MMM_dump_isin.json
options
[MMM] written to ./yf_dumps\MMM_

TypeError: keys must be str, int, float, bool or None, not Timestamp

## EDA

In [ ]:
data.tickers['ABT'].balance_sheet

,2024-12-31,2023-12-31,2022-12-31,2021-12-31,2020-12-31
Treasury Shares Number,4.045628e+08,3.914519e+08,3.947880e+08,3.721876e+08,NaN
Ordinary Shares Number,5.394703e+08,5.525811e+08,5.492451e+08,5.718455e+08,NaN
Share Issued,9.440331e+08,9.440331e+08,9.440331e+08,9.440331e+08,NaN
Net Debt,7.444000e+09,1.030000e+10,1.228400e+10,1.279900e+10,NaN
Total Debt,1.365900e+10,1.675100e+10,1.685500e+10,1.831000e+10,NaN
...,...,...,...,...,...
Allowance For Doubtful Accounts Receivable,-6.000000e+07,-6.200000e+07,-1.740000e+08,-1.890000e+08,NaN
Gross Accounts Receivable,3.254000e+09,3.663000e+09,4.706000e+09,4.849000e+09,NaN
Cash Cash Equivalents And Short Term Investments,7.728000e+09,5.785000e+09,3.893000e+09,4.765000e+09,NaN
Other Short Term Investments,2.128000e+09,5.000000e+07,2.380000e+08,2.010000e+08,NaN


In [18]:
data.tickers['MMM'].cash_flow

,2024-12-31,2023-12-31,2022-12-31,2021-12-31,2020-12-31
Free Cash Flow,6.380000e+08,5.065000e+09,3.842000e+09,5.851000e+09,NaN
Repurchase Of Capital Stock,-1.801000e+09,-3.300000e+07,-1.464000e+09,-2.199000e+09,NaN
Repayment Of Debt,-2.656000e+09,-3.086000e+09,-1.179000e+09,-1.144000e+09,NaN
Issuance Of Debt,8.367000e+09,2.835000e+09,1.000000e+06,1.000000e+06,NaN
Capital Expenditure,-1.181000e+09,-1.615000e+09,-1.749000e+09,-1.603000e+09,NaN
Interest Paid Supplemental Data,5.050000e+08,5.200000e+08,4.400000e+08,4.720000e+08,NaN
Income Tax Paid Supplemental Data,8.520000e+08,1.384000e+09,1.320000e+09,1.695000e+09,NaN
End Cash Position,5.600000e+09,5.933000e+09,3.655000e+09,4.564000e+09,NaN
Beginning Cash Position,5.933000e+09,3.655000e+09,4.564000e+09,4.634000e+09,NaN
Effect Of Exchange Rate Changes,-4.400000e+07,-4.800000e+07,-1.040000e+08,-6.200000e+07,NaN


In [19]:
news = data.news()

In [20]:
list(news.keys())

['MMM', 'AOS', 'ABT']

In [21]:
news['ABT'][6]

{'id': 'f806f0fe-d549-3b9d-994d-9754234502d1',
 'content': {'id': 'f806f0fe-d549-3b9d-994d-9754234502d1',
  'contentType': 'STORY',
  'title': 'CMS covers cardiac ablation in ambulatory surgery centers',
  'description': '',
  'summary': 'Extending Medicare reimbursement for cardiac catheter ablation to settings outside the hospital is expected to boost procedure volumes and benefit Abbott, Boston Scientific, J&J and Medtronic.',
  'pubDate': '2025-11-26T09:58:51Z',
  'displayTime': '2025-11-26T09:58:51Z',
  'isHosted': True,
  'bypassModal': False,
  'previewUrl': None,
  'thumbnail': {'originalUrl': 'https://media.zenfs.com/en/medtech_dive_335/c7f6823802519856c48240b8ba6d6d17',
   'originalWidth': 1600,
   'originalHeight': 900,
   'caption': "Boston Scientific's Farawave pulsed field ablation catheter.",
   'resolutions': [{'url': 'https://s.yimg.com/uu/api/res/1.2/35WrvPVs6U3y34V3zUaNFQ--~B/aD05MDA7dz0xNjAwO2FwcGlkPXl0YWNoeW9u/https://media.zenfs.com/en/medtech_dive_335/c7f68238025

In [22]:
dat = yfinance.Ticker("MSFT")
dat.info

{'address1': 'One Microsoft Way',
 'city': 'Redmond',
 'state': 'WA',
 'zip': '98052-6399',
 'country': 'United States',
 'phone': '425 882 8080',
 'website': 'https://www.microsoft.com',
 'industry': 'Software - Infrastructure',
 'industryKey': 'software-infrastructure',
 'industryDisp': 'Software - Infrastructure',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': "Microsoft Corporation develops and supports software, services, devices, and solutions worldwide. The company's Productivity and Business Processes segment offers Microsoft 365 Commercial, Enterprise Mobility + Security, Windows Commercial, Power BI, Exchange, SharePoint, Microsoft Teams, Security and Compliance, and Copilot; Microsoft 365 Commercial products, such as Windows Commercial on-premises and Office licensed services; Microsoft 365 Consumer products and cloud services, such as Microsoft 365 Consumer subscriptions, Office licensed on-premises, and other consu

In [23]:
dat.analyst_price_targets

{'current': 490.0,
 'high': 730.0,
 'low': 483.0,
 'mean': 625.4096,
 'median': 634.15}